In [ ]:
%pip -q install "pyarrow==18.1.0" "datasets==3.6.0" "transformers==4.55.4" "accelerate==1.10.1" "peft==0.17.1" "bitsandbytes==0.46.1" "trl==0.21.0" "evaluate==0.4.3" "rouge_score" "nltk" "bert-score" "sentence-transformers" "textstat" "sacrebleu" "sentencepiece" "huggingface_hub"

In [ ]:
import os
import gc
import json
import math
import numpy as np
import pandas as pd
import torch
import evaluate
import nltk
import textstat
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForSequenceClassification
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Clinical_NLP_Project"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints", "10K")
ADAPTER_DIR = os.path.join(PROJECT_DIR, "adapters", "10K")
PRED_DIR = os.path.join(PROJECT_DIR, "predictions", "10K")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results", "10K")
RAG_DIR = os.path.join(PROJECT_DIR, "rag", "10K")

for folder in [CHECKPOINT_DIR, ADAPTER_DIR, PRED_DIR, RESULTS_DIR, RAG_DIR]:
    os.makedirs(folder, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for 4-bit bitsandbytes fine-tuning.")

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add your Hugging Face access token to Colab Secrets as HF_TOKEN.")
login(token=HF_TOKEN, add_to_git_credential=False)
del HF_TOKEN

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
DATASET_NAME = "BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track"
SEED = 42
NUM_TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQUENCE_LENGTH = 384
MAX_NEW_TOKENS = 128
NUM_BEAMS = 4

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
dataset = load_dataset(DATASET_NAME)
shuffled = dataset["train"].shuffle(seed=SEED)

train_10k = shuffled.select(range(10000))
validation_data = shuffled.select(range(10000, 10100))
test_data = shuffled.select(range(10100, 10600))

assert len(train_10k) == 10000
assert len(validation_data) == 100
assert len(test_data) == 500

train_indices = set(range(0, 10000))
validation_indices = set(range(10000, 10100))
test_indices = set(range(10100, 10600))
assert train_indices.isdisjoint(validation_indices)
assert train_indices.isdisjoint(test_indices)
assert validation_indices.isdisjoint(test_indices)

print("Training samples:", len(train_10k))
print("Validation samples:", len(validation_data))
print("Test samples:", len(test_data))

In [ ]:
def format_training_example(example):
    return {
        "text": (
            "### Instruction:\n"
            "Simplify the following radiology report into a clear, "
            "patient-friendly explanation. Preserve the medical findings "
            "and do not add information that is not present in the report.\n\n"
            "### Radiology Report:\n"
            f"{example['radiology_report']}\n\n"
            "### Patient-Friendly Summary:\n"
            f"{example['layman_report']}"
        )
    }

train_10k_text = train_10k.map(
    format_training_example,
    remove_columns=train_10k.column_names
)

validation_text = validation_data.map(
    format_training_example,
    remove_columns=validation_data.column_names
)

print("Training records:", len(train_10k_text))
print("Validation records:", len(validation_text))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

compute_dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

print("Tokenizer and quantized base model loaded.")

In [ ]:
sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=False,
    bf16=False,
    logging_steps=50,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=5,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
    max_length=MAX_SEQUENCE_LENGTH,
    overwrite_output_dir=True
)

steps_per_epoch = math.ceil(
    math.ceil(len(train_10k_text) / TRAIN_BATCH_SIZE) / GRADIENT_ACCUMULATION_STEPS
)
expected_training_steps = steps_per_epoch * NUM_TRAIN_EPOCHS


def has_adapter_files(folder):
    if not os.path.isfile(os.path.join(folder, "adapter_config.json")):
        return False
    return any(
        os.path.isfile(os.path.join(folder, filename))
        for filename in ["adapter_model.safetensors", "adapter_model.bin"]
    )


def valid_resume_checkpoint(folder):
    required_files = [
        "trainer_state.json",
        "optimizer.pt",
        "scheduler.pt",
        "rng_state.pth"
    ]
    return (
        has_adapter_files(folder)
        and all(os.path.isfile(os.path.join(folder, filename)) for filename in required_files)
    )

checkpoint_candidates = []
for item in os.listdir(CHECKPOINT_DIR):
    candidate = os.path.join(CHECKPOINT_DIR, item)
    if os.path.isdir(candidate) and item.startswith("checkpoint-"):
        try:
            step = int(item.split("-")[-1])
            checkpoint_candidates.append((step, candidate))
        except ValueError:
            pass

checkpoint_candidates.sort(reverse=True)
latest_valid_checkpoint = next(
    (path for _, path in checkpoint_candidates if valid_resume_checkpoint(path)),
    None
)

adapter_ready = has_adapter_files(ADAPTER_DIR)
checkpoint_state = {}
checkpoint_is_complete = False
resume_checkpoint = None
completed_adapter_source = ADAPTER_DIR if adapter_ready else None

if latest_valid_checkpoint:
    with open(os.path.join(latest_valid_checkpoint, "trainer_state.json"), "r") as f:
        checkpoint_state = json.load(f)
    checkpoint_step = int(checkpoint_state.get("global_step", 0))
    checkpoint_epoch = float(checkpoint_state.get("epoch") or 0)
    checkpoint_is_complete = (
        checkpoint_step >= expected_training_steps
        or checkpoint_epoch >= NUM_TRAIN_EPOCHS
    )
    if not adapter_ready and checkpoint_is_complete:
        completed_adapter_source = latest_valid_checkpoint
    elif not adapter_ready and not checkpoint_is_complete:
        resume_checkpoint = latest_valid_checkpoint

trainer = None
train_output = None
training_log_history = []

if completed_adapter_source:
    model = PeftModel.from_pretrained(
        base_model,
        completed_adapter_source,
        is_trainable=False
    )
    if completed_adapter_source != ADAPTER_DIR:
        model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
        tokenizer.save_pretrained(ADAPTER_DIR)
    training_log_history = checkpoint_state.get("log_history", [])
    training_status = "loaded_existing_adapter" if completed_adapter_source == ADAPTER_DIR else "loaded_completed_checkpoint"
    training_metrics = {
        "status": training_status,
        "global_step": checkpoint_state.get("global_step"),
        "epoch": checkpoint_state.get("epoch"),
        "log_history": training_log_history
    }
else:
    model = get_peft_model(base_model, lora_config)
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_10k_text,
        eval_dataset=validation_text,
        processing_class=tokenizer
    )
    train_output = trainer.train(resume_from_checkpoint=resume_checkpoint)
    trainer.save_model(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    training_log_history = trainer.state.log_history
    training_status = "resumed_from_checkpoint" if resume_checkpoint else "trained_from_scratch"
    training_metrics = dict(train_output.metrics)
    training_metrics["status"] = training_status
    training_metrics["global_step"] = trainer.state.global_step
    training_metrics["epoch"] = trainer.state.epoch
    training_metrics["log_history"] = training_log_history

training_metrics_path = os.path.join(RESULTS_DIR, "training_metrics_10K.json")
training_history_path = os.path.join(RESULTS_DIR, "training_log_history_10K.csv")

if training_status == "loaded_existing_adapter" and not training_log_history:
    if os.path.isfile(training_metrics_path):
        with open(training_metrics_path, "r") as f:
            training_metrics = json.load(f)
    if os.path.isfile(training_history_path):
        training_log_history = pd.read_csv(training_history_path).to_dict("records")

with open(training_metrics_path, "w") as f:
    json.dump(training_metrics, f, indent=2, default=str)

if training_log_history:
    pd.DataFrame(training_log_history).to_csv(training_history_path, index=False)

print("Training status:", training_status)
print("Adapter saved at:", ADAPTER_DIR)
print("Training metrics saved at:", training_metrics_path)

if trainer is not None:
    del trainer
del model
del base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

generation_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)
generation_model.eval()

print("Saved 10K adapter loaded.")

In [ ]:
radiology_reports = test_data["radiology_report"]
references = test_data["layman_report"]

PREDICTION_PROMPT = """You are a medical communication assistant.

Convert the following radiology report into a clear, simple, patient-friendly explanation.

Rules:
1. Preserve all medical findings stated in the report.
2. Do not add any information that is not stated in the report.
3. Do not add symptoms, diagnoses, causes, treatments, recommendations, or prognosis.
4. Preserve negation, uncertainty, severity, location, and laterality.
5. Explain medical terms in simple language when needed.
6. Do not remove important findings.
7. Do not introduce information from outside the given report.
8. Return only the patient-friendly explanation.

Radiology Report:
{report}

Patient-Friendly Explanation:
"""


def generate_summary(report):
    prompt = PREDICTION_PROMPT.format(report=report)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=384
    ).to(generation_model.device)

    with torch.no_grad():
        output = generation_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    stop_phrases = [
        "\nRules:",
        "\n1. Preserve",
        "\n2. Do not",
        "\n3. Do not",
        "\n4. Preserve",
        "\n5. Explain",
        "\n6. Do not",
        "\n7. Do not",
        "\n8. Return"
    ]

    for phrase in stop_phrases:
        if phrase in generated_text:
            generated_text = generated_text.split(phrase)[0].strip()

    return generated_text


generated = []
for report in tqdm(radiology_reports, total=len(radiology_reports), desc="10K predictions"):
    generated.append(generate_summary(report))

if len(generated) != 500:
    raise RuntimeError(f"Expected 500 generated summaries, received {len(generated)}.")

prediction_path = os.path.join(PRED_DIR, "predictions_10K_test_500.csv")
predictions_10k_df = pd.DataFrame({
    "radiology_report": radiology_reports,
    "reference_layman_summary": references,
    "generated_summary": generated
})
predictions_10k_df.to_csv(prediction_path, index=False)

print("Saved 500 predictions:", prediction_path)

del generation_model
del base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

prediction_path = os.path.join(PRED_DIR, "predictions_10K_test_500.csv")
prediction_df = pd.read_csv(prediction_path)

if len(prediction_df) != 500:
    raise ValueError(f"Expected 500 saved predictions, found {len(prediction_df)}.")

required_columns = [
    "radiology_report",
    "reference_layman_summary",
    "generated_summary"
]
if not all(column in prediction_df.columns for column in required_columns):
    raise ValueError("The saved prediction CSV does not contain the required columns.")

df100 = prediction_df.iloc[:100].copy()
reports = df100["radiology_report"].fillna("").astype(str).tolist()
references = df100["reference_layman_summary"].fillna("").astype(str).tolist()
predictions = df100["generated_summary"].fillna("").astype(str).tolist()

if len(df100) != 100:
    raise ValueError(f"Expected 100 evaluation rows, found {len(df100)}.")

pred100_path = os.path.join(PRED_DIR, "predictions_10K_100_samples.csv")
df100[required_columns].to_csv(pred100_path, index=False)

bleu_metric = evaluate.load("bleu")
bleu_result = bleu_metric.compute(
    predictions=predictions,
    references=[[reference] for reference in references]
)
bleu_score = float(bleu_result["bleu"])

rouge_metric = evaluate.load("rouge")
rouge_result = rouge_metric.compute(
    predictions=predictions,
    references=references
)
rouge1 = float(rouge_result["rouge1"])
rouge2 = float(rouge_result["rouge2"])
rougeL = float(rouge_result["rougeL"])

meteor_metric = evaluate.load("meteor")
meteor_result = meteor_metric.compute(
    predictions=predictions,
    references=references
)
meteor = float(meteor_result["meteor"])

bertscore_metric = evaluate.load("bertscore")
bert_result = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    lang="en"
)
bert_precision = float(np.mean(bert_result["precision"]))
bert_recall = float(np.mean(bert_result["recall"]))
bert_f1 = float(np.mean(bert_result["f1"]))

del bertscore_metric

gc.collect()
torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
reference_embeddings = embedding_model.encode(
    references,
    normalize_embeddings=True,
    show_progress_bar=True
)
prediction_embeddings = embedding_model.encode(
    predictions,
    normalize_embeddings=True,
    show_progress_bar=True
)
semantic_scores = np.sum(
    reference_embeddings * prediction_embeddings,
    axis=1
)
semantic_similarity = float(np.mean(semantic_scores))

del embedding_model
gc.collect()
torch.cuda.empty_cache()

fkgl = float(np.mean([textstat.flesch_kincaid_grade(text) for text in predictions]))
dale_chall = float(np.mean([textstat.dale_chall_readability_score(text) for text in predictions]))
coleman_liau = float(np.mean([textstat.coleman_liau_index(text) for text in predictions]))

metrics_100 = {
    "Evaluation Samples": 100,
    "BLEU": bleu_score,
    "ROUGE-1": rouge1,
    "ROUGE-2": rouge2,
    "ROUGE-L": rougeL,
    "METEOR": meteor,
    "BERTScore Precision": bert_precision,
    "BERTScore Recall": bert_recall,
    "BERTScore F1": bert_f1,
    "Semantic Similarity": semantic_similarity,
    "FKGL": fkgl,
    "Dale-Chall": dale_chall,
    "Coleman-Liau": coleman_liau
}

metrics_csv = os.path.join(RESULTS_DIR, "METRICS_FINAL_100_10K.csv")
metrics_json = os.path.join(RESULTS_DIR, "metrics_final_100_10K.json")
metrics_df = pd.DataFrame(
    [{"Metric": metric, "10K Score": score} for metric, score in metrics_100.items()]
)
metrics_df.to_csv(metrics_csv, index=False)

with open(metrics_json, "w") as f:
    json.dump(metrics_100, f, indent=4)

print(metrics_df.to_string(index=False))
print("100-row predictions:", pred100_path)
print("Metrics CSV:", metrics_csv)
print("Metrics JSON:", metrics_json)

In [ ]:
NLI_MODEL_NAME = "microsoft/deberta-base-mnli"
NLI_RESULTS_PATH = os.path.join(RESULTS_DIR, "NLI_RESULTS_100_10K.json")

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(nli_device)
nli_model.eval()

id2label = {
    int(key): str(value).lower()
    for key, value in nli_model.config.id2label.items()
}

contradiction_id = next(
    (index for index, label in id2label.items() if "contradiction" in label),
    None
)
neutral_id = next(
    (index for index, label in id2label.items() if "neutral" in label),
    None
)
entailment_id = next(
    (index for index, label in id2label.items() if "entailment" in label),
    None
)

if None in [contradiction_id, neutral_id, entailment_id]:
    raise ValueError(f"Could not identify all NLI labels from model configuration: {id2label}")

contradiction_count = 0
neutral_count = 0
entailment_count = 0
batch_size = 4

with torch.no_grad():
    for start in range(0, len(df100), batch_size):
        batch = df100.iloc[start:start + batch_size]
        premises = batch["radiology_report"].fillna("").astype(str).tolist()
        hypotheses = batch["generated_summary"].fillna("").astype(str).tolist()

        nli_inputs = nli_tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        nli_inputs = {key: value.to(nli_device) for key, value in nli_inputs.items()}
        logits = nli_model(**nli_inputs).logits
        batch_predictions = torch.argmax(logits, dim=-1).cpu().tolist()

        for predicted_label in batch_predictions:
            if predicted_label == contradiction_id:
                contradiction_count += 1
            elif predicted_label == neutral_id:
                neutral_count += 1
            elif predicted_label == entailment_id:
                entailment_count += 1

total_nli = len(df100)
entailment_rate = entailment_count / total_nli * 100
neutral_rate = neutral_count / total_nli * 100
contradiction_rate = contradiction_count / total_nli * 100

nli_results = {
    "evaluation": "10K NLI-based factual consistency proxy",
    "model": NLI_MODEL_NAME,
    "evaluation_samples": int(total_nli),
    "premise": "Original radiology report",
    "hypothesis": "Generated patient-friendly summary",
    "entailment_count": int(entailment_count),
    "neutral_count": int(neutral_count),
    "contradiction_count": int(contradiction_count),
    "entailment_rate_percent": round(entailment_rate, 6),
    "neutral_rate_percent": round(neutral_rate, 6),
    "contradiction_rate_percent": round(contradiction_rate, 6),
    "contradiction_based_nli_proxy_rate_percent": round(contradiction_rate, 6),
    "interpretation": "The contradiction percentage is an NLI-based proxy only. It is not confirmed hallucination detection, clinical validation, or expert clinical evaluation."
}

with open(NLI_RESULTS_PATH, "w") as f:
    json.dump(nli_results, f, indent=4)

metrics_100["NLI Entailment Count"] = int(entailment_count)
metrics_100["NLI Neutral Count"] = int(neutral_count)
metrics_100["NLI Contradiction Count"] = int(contradiction_count)
metrics_100["NLI Entailment Rate (%)"] = round(entailment_rate, 6)
metrics_100["NLI Neutral Rate (%)"] = round(neutral_rate, 6)
metrics_100["NLI Contradiction Rate (%)"] = round(contradiction_rate, 6)
metrics_100["Contradiction-Based NLI Proxy Rate (%)"] = round(contradiction_rate, 6)
metrics_100["NLI Model"] = NLI_MODEL_NAME

metrics_df = pd.DataFrame(
    [{"Metric": metric, "10K Score": score} for metric, score in metrics_100.items()]
)
metrics_df.to_csv(metrics_csv, index=False)

with open(metrics_json, "w") as f:
    json.dump(metrics_100, f, indent=4)

print("Entailment:", entailment_count, f"({entailment_rate:.2f}%)")
print("Neutral:", neutral_count, f"({neutral_rate:.2f}%)")
print("Contradiction:", contradiction_count, f"({contradiction_rate:.2f}%)")
print("NLI-based proxy results:", NLI_RESULTS_PATH)
print("Updated metrics:", metrics_csv)
print("Updated metrics JSON:", metrics_json)

del nli_model
del nli_tokenizer
gc.collect()
torch.cuda.empty_cache()